# 055 — Offline RL & imitation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/055-offline-rl-and-imitation.ipynb)

Offline learning sees only a fixed dataset. This notebook computes discounted returns backwards, shows how per-step accuracy compounds over a rollout, and makes the invented Q-value for an unseen action visible.

## 1. Returns, backwards

`G_t = r_t + gamma * G_{t+1}`. Start at the last reward and walk left; starting at the front builds each return from the wrong tail.

In [ ]:
import numpy as np                              # array library, preinstalled on Colab

rewards = np.array([0.0, 1.0, 1.0])             # three steps
gamma = 0.9                                     # discount per step
returns = np.zeros(3)                           # one G per step
total = 0.0                                     # running tail value

for t in range(len(rewards) - 1, -1, -1):       # last step first
    total = rewards[t] + gamma * total          # add the discounted tail
    returns[t] = total                          # store G_t

print(returns.round(2))                         # [1.71 1.9 1.0]

## 2. Compounding error and an invented Q

A policy that is right 90% per step is not 90% successful over ten steps. And a `max` over actions the dataset never contains can invent a value that beats every real one.

In [ ]:
import matplotlib.pyplot as plt                 # plotting library, preinstalled on Colab

steps = np.arange(1, 11)                        # horizons 1 to 10
for acc in (0.80, 0.90, 0.95):                  # three per-step accuracies
    print(acc, round(acc ** 10, 3))             # 0.107, 0.349, 0.599

data_actions = {"a1": 2.0, "a2": 2.2}          # only these appear in D
invented = 5.0                                  # Q(s', a3) for an unseen action
target_seen = 0.0 + 0.9 * max(data_actions.values())   # 1.98
target_unseen = 0.0 + 0.9 * invented                   # 4.50
print("target with support:", target_seen, "without:", target_unseen)

plt.figure(figsize=(6, 3))                      # wide, short figure
plt.plot(steps, 0.9 ** steps, marker="o", label="90% per step")
plt.plot(steps, 0.8 ** steps, marker="s", label="80% per step")
plt.xlabel("steps in the rollout")              # x-axis
plt.ylabel("probability still correct")         # y-axis
plt.legend()                                    # name the lines
plt.title("small per-step errors compound")     # takeaway
plt.show()                                      # render

## Your turn
Change `gamma` to 0.5 in section 1 and predict all three returns before rerunning. Then set `invented = 2.1`: the unseen action no longer beats the data, and the `max` stops pulling the policy outside the support. What would a CQL penalty add on top?